In [ ]:
import sys
sys.path.insert(0, "..")  # if running from notebooks/

from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

from hanoi_caption.kb_loader import load_kb
from hanoi_caption.kb_indexer import build_or_load_index
from hanoi_caption.image_describer import describe_image
from hanoi_caption.kb_matcher import match_kb
from hanoi_caption.composer import compose
from hanoi_caption.pipeline import caption_phase1
from hanoi_caption.model_registry import registry

In [ ]:
nodes = load_kb("../data/kb.json")
print(f"Loaded {len(nodes)} KB nodes:")
for nid, n in nodes.items():
    print(f"  {nid:35s} {n.name_en}")
kb_index = build_or_load_index(nodes)
print(f"\nKB index ready, {kb_index.embeddings.shape}")

In [ ]:
test_image = Path("../tests/fixtures/temple_of_literature_1.jpg")
img = Image.open(test_image).convert("RGB")
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
result = caption_phase1(image=img, kb_nodes=nodes, kb_index=kb_index)
print("=== DEBUG ===")
import json
print(json.dumps(result.debug, indent=2, default=str)[:2000])
print("\n=== OUTPUT ===")
if result.caption:
    print(result.caption)
else:
    print("REFUSED:", result.refusal)

In [ ]:
# Use any non-Hanoi photo (e.g., your local park, kitchen, etc.)
neg_image = Path("../tests/fixtures/non_hanoi.jpg")
if neg_image.exists():
    nimg = Image.open(neg_image).convert("RGB")
    nres = caption_phase1(image=nimg, kb_nodes=nodes, kb_index=kb_index)
    print("expected refusal:", nres.refusal or "got caption (unexpected)")
else:
    print("Skipped — drop a non-Hanoi photo at", neg_image)

In [ ]:
import torch
print("loaded models:", registry.loaded())
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"VRAM reserved : {torch.cuda.memory_reserved()/1e9:.2f} GB")